In [1]:
# =============================================================================
# STEP 8 - CLASS-BUDGET-MATCHED TSC vs SHC
#
# WHY THIS IS THE MOST IMPORTANT REMAINING EXPERIMENT.
#
# Section 4.15 states that the study's CONFIRMATORY content is the protocol
# contrast and the per-class coverage outcomes. Everything else is exploratory.
# But Section 5.12 concedes that TSC and SHC differ in per-class calibration
# BUDGET as well as in calibration-data provenance: on NSL-KDD the focal class
# gets 149 source points under SHC against 297 target points under TSC. Under
# exchangeability a smaller calibration set biases towards over-coverage, but once
# exchangeability is lost that argument fails and the direction is unknown.
#
# So the single confirmatory claim currently rests on a comparison confounded with
# unknown sign. This notebook removes the confound by subsampling both calibration
# sets to an IDENTICAL per-class count, and reports both contrasts:
#
#   natural-budget   : what an operator actually faces  (current results)
#   budget-matched   : provenance effect with support held fixed  (new)
#
# IT CAN GO AGAINST THE PAPER. If the gap collapses under matching, the headline
# result is substantially a sample-size artefact and the paper must say so.
# =============================================================================
import numpy as np, pandas as pd
from scipy import stats
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
assert PROJECT_ROOT.exists(), 'Drive mount unhealthy; restart runtime and remount'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
from conformal import conformal_q
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY; R=getattr(config,'N_MATCHED_DRAWS',10)
def dseed(*p): return int(hashlib.sha256('|'.join(map(str,p)).encode()).hexdigest(),16)%(2**32)
print('ready | alpha', ALPHA, '| draws', R)


Mounted at /content/drive
ready | alpha 0.05 | draws 10


In [2]:
# =============================================================================
# Cell 2 - the matching rule and the two-protocol evaluator.
#
# For each class k: n_match[k] = min(n_SHC[k], n_TSC[k]), then BOTH calibration
# sets are subsampled to exactly n_match[k] points of that class before the
# Mondrian quantile is formed. Everything else, including the evaluation set, the
# classifier, the calibrator and the realised scores, is identical to the
# natural-budget run. The ONLY thing that changes is per-class calibration size.
# =============================================================================
def aps_scores(P, rng):
    o=np.argsort(-P,axis=1); sp=np.take_along_axis(P,o,1); cum=np.cumsum(sp,1)
    U=rng.random(len(P))[:,None]; ss=cum-(1-U)*sp
    out=np.empty_like(P); np.put_along_axis(out,o,ss,1); return out

def mondrian_q(S, y, K, alpha):
    tc=S[np.arange(len(y)), y]
    q=np.full(K, np.inf); n=np.zeros(K, int)
    for k in range(K):
        s=tc[y==k]; n[k]=len(s)
        q[k]=conformal_q(s, alpha)[0]
    return q, n

def match_indices(y_a, y_b, K, rng):
    """Return per-class index subsets of a and b with identical class counts."""
    ia, ib, sizes = [], [], {}
    for k in range(K):
        ka=np.where(y_a==k)[0]; kb=np.where(y_b==k)[0]
        m=min(len(ka), len(kb)); sizes[k]=m
        if m==0: continue
        ia.append(rng.choice(ka, m, replace=False))
        ib.append(rng.choice(kb, m, replace=False))
    return (np.concatenate(ia) if ia else np.array([],int),
            np.concatenate(ib) if ib else np.array([],int), sizes)

def coverage_rows(S_ev, y_ev, q, ncal, K, names, tag, extra):
    inset = S_ev <= q[None,:]
    cov = inset[np.arange(len(y_ev)), y_ev]
    out=[]
    for k in range(K):
        m=y_ev==k
        if not m.any(): continue
        out.append({'class':names[k],'budget':tag,'coverage':float(cov[m].mean()),
                    'n_eval':int(m.sum()),'n_cal':int(ncal[k]),
                    'set_size':float(inset[m].sum(1).mean()),
                    'feasible':bool(np.isfinite(q[k])), **extra})
    return out
print('matching rule and evaluator defined')
print('  n_match[k] = min(n_SHC[k], n_TSC[k]); both protocols subsampled to it')


matching rule and evaluator defined
  n_match[k] = min(n_SHC[k], n_TSC[k]); both protocols subsampled to it


In [3]:
# =============================================================================
# Cell 3 - NSL-KDD at rung 0.80, where the documented asymmetry is 149 vs 297.
# =============================================================================
CL=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CL)}; K=len(CL)
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
y_sp=tr[tr.partition=='source_cal_pool']['label'].map(c2i).to_numpy()
y_te=te['label'].map(c2i).to_numpy()
assign=pd.read_parquet(config.PROC_DIR/'nslkdd_ladder_assignments.parquet')
IDX={(r,j,role):g['test_idx'].to_numpy() for (r,j,role),g in assign.groupby(['rung','realization','role'])}
RUNG=0.80; REALS=sorted(assign[np.isclose(assign.rung,RUNG)]['realization'].unique())
rows=[]; t0=time.time()
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    d=np.load(f); P_sp=d['S_pool'].astype(np.float64); P_te=d['target'].astype(np.float64)
    for j in REALS:
        ev=IDX.get((RUNG,j,'eval')); tc=IDX.get((RUNG,j,'tcal'))
        if ev is None or tc is None or len(ev)==0 or len(tc)==0: continue
        for draw in range(R):
            rng=np.random.default_rng(dseed('nslmatch',arch,seed,j,draw))
            S_ev=aps_scores(P_te[ev], np.random.default_rng(dseed('nslm-e',arch,seed,j,draw)))
            S_tc=aps_scores(P_te[tc], np.random.default_rng(dseed('nslm-t',arch,seed,j,draw)))
            S_sc=aps_scores(P_sp,     np.random.default_rng(dseed('nslm-s',arch,seed,j,draw)))
            y_ev=y_te[ev]; y_tc=y_te[tc]
            ex={'dataset':'nslkdd','arch':arch,'seed':seed,'realization':int(j),'draw':draw}
            # natural budget
            for tag,(S,yy) in {'TSC':(S_tc,y_tc),'SHC':(S_sc,y_sp)}.items():
                q,nc=mondrian_q(S,yy,K,ALPHA)
                rows += coverage_rows(S_ev,y_ev,q,nc,K,CL,'natural',{**ex,'protocol':tag})
            # matched budget
            ia,ib,sizes=match_indices(y_tc,y_sp,K,rng)
            qT,ncT=mondrian_q(S_tc[ia],y_tc[ia],K,ALPHA)
            qS,ncS=mondrian_q(S_sc[ib],y_sp[ib],K,ALPHA)
            rows += coverage_rows(S_ev,y_ev,qT,ncT,K,CL,'matched',{**ex,'protocol':'TSC'})
            rows += coverage_rows(S_ev,y_ev,qS,ncS,K,CL,'matched',{**ex,'protocol':'SHC'})
NSL=pd.DataFrame(rows)
print(f'NSL rows {len(NSL):,} | {time.time()-t0:.0f}s')
chk=NSL[NSL.budget=='matched'].groupby(['class','protocol'])['n_cal'].mean().unstack()
print('\nmatched per-class calibration counts (TSC vs SHC must be equal):')
print(chk.round(1).to_string())
assert np.allclose(chk['TSC'], chk['SHC']), 'matching failed: counts differ'
print('matching verified')


NSL rows 120,000 | 63s

matched per-class calibration counts (TSC vs SHC must be equal):
protocol     SHC     TSC
class                   
DoS        771.8   771.8
Normal    1008.0  1008.0
Probe      249.6   249.6
R2L        149.0   149.0
U2R          5.7     5.7
matching verified


In [4]:
# =============================================================================
# Cell 4 - UGR'16 and CIC-IoT-2023, same construction.
# =============================================================================
def run_env(name, Psp_files, y_sp, y_tg, get_probs, classes, eval_idx_fn=None):
    K=len(classes); out=[]
    for f in Psp_files:
        arch,seed,Psp,Ptg = get_probs(f)
        mm=min(len(y_sp), len(y_tg)//2)
        for draw in range(R):
            rng=np.random.default_rng(dseed(name,seed,arch,draw))
            tp=rng.permutation(len(y_tg)); de,tc = tp[:mm], tp[mm:2*mm]
            sc=rng.permutation(len(y_sp))[:mm]
            S_ev=aps_scores(Ptg[de], np.random.default_rng(dseed(name,seed,arch,draw,'e')))
            S_tc=aps_scores(Ptg[tc], np.random.default_rng(dseed(name,seed,arch,draw,'t')))
            S_sc=aps_scores(Psp[sc], np.random.default_rng(dseed(name,seed,arch,draw,'s')))
            y_ev,y_tcal,y_scal = y_tg[de], y_tg[tc], y_sp[sc]
            ex={'dataset':name,'arch':arch,'seed':seed,'realization':0,'draw':draw}
            for tag,(S,yy) in {'TSC':(S_tc,y_tcal),'SHC':(S_sc,y_scal)}.items():
                q,nc=mondrian_q(S,yy,K,ALPHA)
                out += coverage_rows(S_ev,y_ev,q,nc,K,classes,'natural',{**ex,'protocol':tag})
            ia,ib,_=match_indices(y_tcal,y_scal,K,rng)
            qT,ncT=mondrian_q(S_tc[ia],y_tcal[ia],K,ALPHA)
            qS,ncS=mondrian_q(S_sc[ib],y_scal[ib],K,ALPHA)
            out += coverage_rows(S_ev,y_ev,qT,ncT,K,classes,'matched',{**ex,'protocol':'TSC'})
            out += coverage_rows(S_ev,y_ev,qS,ncS,K,classes,'matched',{**ex,'protocol':'SHC'})
    return out

# ---- UGR'16 ----
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
y_sp_u=us[us.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
y_tg_u=ut['label'].map(U2I).to_numpy()
def gp_ugr(f):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f)
    return arch,int(sd.replace('seed','')),d['srcpool'],d['target']
UGRrows=run_env('ugr16', sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')),
                y_sp_u, y_tg_u, gp_ugr, UCL)
print('UGR rows', len(UGRrows))

# ---- CIC-IoT-2023 ----
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
ic2i={c:i for i,c in enumerate(ICL)}
iot['y']=iot['family'].map(ic2i).astype(np.int64)
y_sp_i=iot.loc[iot.partition=='source_cal_pool','y'].to_numpy()
y_tg_i=iot.loc[iot.partition=='target_pool','y'].to_numpy()
def gp_iot(f):
    _,arch,sd=Path(f).stem.split('__'); d=np.load(f)
    return arch,int(sd.replace('seed','')),d['srcpool'],d['target']
IOTrows=run_env('ciciot2023', sorted((config.DATA_DIR/'ciciot_probs').glob('ciciot2023__*.npz')),
                y_sp_i, y_tg_i, gp_iot, ICL)
print('CIC-IoT rows', len(IOTrows))
ALL=pd.concat([NSL, pd.DataFrame(UGRrows), pd.DataFrame(IOTrows)], ignore_index=True)
ALL=ALL[ALL.feasible]
print('\ntotal feasible rows', len(ALL))


UGR rows 6000
CIC-IoT rows 9600

total feasible rows 111600


In [5]:
# =============================================================================
# Cell 5 - THE TEST. Does the TSC-SHC gap survive matching the class budgets?
# =============================================================================
FOCAL={'nslkdd':'R2L','ugr16':'nerisbotnet','ciciot2023':'Web'}
print('FOCAL-CLASS COVERAGE, natural vs budget-matched')
print(f"{'dataset':12s} {'budget':9s} {'TSC':>8s} {'SHC':>8s} {'gap':>8s} {'n_cal TSC':>10s} {'n_cal SHC':>10s}")
res=[]
for ds,fc in FOCAL.items():
    g=ALL[(ALL.dataset==ds)&(ALL['class']==fc)]
    if not len(g): continue
    for b in ['natural','matched']:
        gb=g[g.budget==b]
        t=gb[gb.protocol=='TSC']; s=gb[gb.protocol=='SHC']
        if not len(t) or not len(s): continue
        gap=float(t.coverage.mean()-s.coverage.mean())
        print(f"{ds:12s} {b:9s} {t.coverage.mean():8.4f} {s.coverage.mean():8.4f} "
              f"{gap:8.4f} {t.n_cal.mean():10.1f} {s.n_cal.mean():10.1f}")
        res.append({'dataset':ds,'class':fc,'budget':b,'TSC':t.coverage.mean(),
                    'SHC':s.coverage.mean(),'gap':gap,
                    'n_cal_TSC':t.n_cal.mean(),'n_cal_SHC':s.n_cal.mean()})
RES=pd.DataFrame(res)

print('\nHOW MUCH OF THE GAP SURVIVES MATCHING?')
for ds in FOCAL:
    r=RES[RES.dataset==ds]
    if len(r)==2:
        nat=float(r[r.budget=='natural'].gap.iloc[0]); mat=float(r[r.budget=='matched'].gap.iloc[0])
        frac=mat/nat if abs(nat)>1e-9 else np.nan
        print(f"  {ds:12s} natural {nat:+.4f} -> matched {mat:+.4f}   retained {frac:.1%}")

# seed-clustered interval on the MATCHED gap for the primary environment
rng=np.random.default_rng(20260726)
g=ALL[(ALL.dataset=='nslkdd')&(ALL['class']=='R2L')&(ALL.budget=='matched')]
seeds=g['seed'].unique()
per=np.array([g[(g.seed==s)&(g.protocol=='TSC')].coverage.mean()
              -g[(g.seed==s)&(g.protocol=='SHC')].coverage.mean() for s in seeds])
bs=[rng.choice(per,len(per),replace=True).mean() for _ in range(4000)]
print(f"\nNSL-KDD matched focal gap: {per.mean():.4f}  95% CI [{np.percentile(bs,2.5):.4f}, {np.percentile(bs,97.5):.4f}]")

print('\nVERDICT:')
r=RES[RES.dataset=='nslkdd']
nat=float(r[r.budget=='natural'].gap.iloc[0]); mat=float(r[r.budget=='matched'].gap.iloc[0])
if abs(mat) >= 0.8*abs(nat):
    print('  The gap SURVIVES budget matching. The confirmatory contrast is a provenance')
    print('  effect, not a sample-size artefact, and the confound identified in Section 5.12')
    print('  is resolved rather than merely acknowledged.')
elif abs(mat) >= 0.4*abs(nat):
    print('  The gap is REDUCED but substantial. Report both contrasts; the natural-budget')
    print('  figure overstates the pure provenance effect and must not be quoted alone.')
else:
    print('  The gap LARGELY COLLAPSES under matching. The headline result is substantially a')
    print('  calibration-size artefact. The paper must lead with the matched contrast and')
    print('  restate its central claim accordingly.')


FOCAL-CLASS COVERAGE, natural vs budget-matched
dataset      budget         TSC      SHC      gap  n_cal TSC  n_cal SHC
nslkdd       natural     0.9527   0.0290   0.9237      296.9      149.0
nslkdd       matched     0.9530   0.0290   0.9240      149.0      149.0
ugr16        natural     0.9495   0.9471   0.0024     7505.0     7500.0
ugr16        matched     0.9495   0.9472   0.0023     7470.9     7470.9
ciciot2023   natural     0.9496   0.9515  -0.0019     3665.0     2135.0
ciciot2023   matched     0.9500   0.9515  -0.0015     2135.0     2135.0

HOW MUCH OF THE GAP SURVIVES MATCHING?
  nslkdd       natural +0.9237 -> matched +0.9240   retained 100.0%
  ugr16        natural +0.0024 -> matched +0.0023   retained 96.6%
  ciciot2023   natural -0.0019 -> matched -0.0015   retained 79.3%

NSL-KDD matched focal gap: 0.9240  95% CI [0.9217, 0.9265]

VERDICT:
  The gap SURVIVES budget matching. The confirmatory contrast is a provenance
  effect, not a sample-size artefact, and the confound ide

In [ ]:
# =============================================================================
# Cell 6 - save and commit
# =============================================================================
ALL.to_csv(RD/'budget_matched_cells.csv', index=False)
RES.to_csv(RD/'budget_matched_summary.csv', index=False)
(RD/'budget_matched_verdict.json').write_text(json.dumps({
 'why':'the study\u2019s confirmatory contrast (TSC vs SHC) was confounded with per-class '
       'calibration budget, of unknown sign once exchangeability is lost',
 'matching':'n_match[k] = min(n_SHC[k], n_TSC[k]); both calibration sets subsampled to it '
            'per class; evaluation set, classifier, calibrator and realised scores unchanged',
 'summary':RES.round(5).to_dict('records'),
 'nsl_matched_gap':float(per.mean()),
 'nsl_matched_ci':[float(np.percentile(bs,2.5)),float(np.percentile(bs,97.5))]},
 indent=2, default=str))
print('saved budget_matched_{cells,summary}.csv and the verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT)
lock=PROJECT_ROOT/'.git'/'index.lock'
if lock.exists() and not subprocess.run(['pgrep','git'],capture_output=True).stdout.strip():
    lock.unlink(); print('removed stale git lock')
for attempt in (1,2):
    git('add','-A',show=False)
    if git('status','--porcelain',show=False).stdout.strip():
        git('commit','-m','step 8: class-budget-matched TSC vs SHC; resolves the calibration-size confound on the confirmatory contrast')
        r=git('push','-u','origin','main')
        if r.returncode: print('PUSH FAILED. Commit is safe locally.')
        break
    if attempt==1: print('waiting 10s for Drive sync...'); time.sleep(10)
    else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


saved budget_matched_{cells,summary}.csv and the verdict
